# Sanity check: FastPM L2000-N256 runs (2000, 2001, 2002 vs benchmark 0)

Quick visual/quantitative validation before launching the full lhid 2000-3999 suite.
Each run has a different cosmology + ICs, so we only check that fields are **well-formed**
and **physically reasonable** — not that they match each other.

Checks: (1) structure & dimensionality, (2) `rho`/`fvel` basic stats, (3) density slices,
(4) 1-pt PDF of δ, (5) 2-pt power spectrum, (6) velocity sanity, (7) growth across snapshots,
(8) comparison against CAMB linear theory.

Run with the **`cmass`** conda env (needs `Pk_library` from Pylians and `camb`).

In [ ]:
import os
from os.path import join
import numpy as np
import h5py
import yaml
import matplotlib.pyplot as plt
import Pk_library as PKL

BASEDIR = '/work/hdd/bdne/maho3/cmass-ili/abacuslike/fastpm/L2000-N256'
L = 2000.0   # Mpc/h
N = 256

# label -> lhid directory
RUNS = {
    '0 (benchmark)': '0',
    '2000': '2000',
    '2001': '2001',
    '2002': '2002',
}
COLORS = {k: c for k, c in zip(RUNS, ['k', 'C0', 'C1', 'C2'])}
print('Runs:', list(RUNS))

In [ ]:
# --- helpers ---------------------------------------------------------------
def run_path(lhid):
    return join(BASEDIR, lhid)


def load_cosmo(lhid):
    """Return [Omega_m, Omega_b, h, n_s, sigma8] from the run's config.yaml."""
    with open(join(run_path(lhid), 'config.yaml')) as f:
        cfg = yaml.safe_load(f)
    return cfg['nbody']['cosmo']


def scale_factors(lhid):
    """Sorted list of snapshot scale-factor group keys (as strings)."""
    with h5py.File(join(run_path(lhid), 'nbody.h5'), 'r') as f:
        return sorted(f.keys(), key=float)


def load_field(lhid, a_key, name):
    """Load 'rho' (N,N,N) or 'fvel' (N,N,N,3) for a given snapshot key."""
    with h5py.File(join(run_path(lhid), 'nbody.h5'), 'r') as f:
        return f[a_key][name][:]


def compute_pk(delta, BoxSize=L, MAS='CIC', threads=4):
    """Matter P(k) of an overdensity field via Pylians."""
    pk = PKL.Pk(delta.astype(np.float32), BoxSize, axis=0, MAS=MAS,
                verbose=False, threads=threads)
    return pk.k3D, pk.Pk[:, 0]


def projected_slab(field3d, axis=2, thickness=8):
    """Mean of a thin slab along `axis` -> 2D map."""
    sl = [slice(None)] * 3
    sl[axis] = slice(0, thickness)
    return field3d[tuple(sl)].mean(axis=axis)

## 1. Structure, dimensionality & cosmology

In [ ]:
for label, lhid in RUNS.items():
    p = join(run_path(lhid), 'nbody.h5')
    assert os.path.isfile(p), f'MISSING: {p}'
    aks = scale_factors(lhid)
    cosmo = load_cosmo(lhid)
    rho = load_field(lhid, aks[-1], 'rho')
    fvel = load_field(lhid, aks[-1], 'fvel')
    print(f'{label:>14}  | n_snap={len(aks):2d}  a=[{aks[0]},...,{aks[-1]}]')
    print(f'{"":>14}  | rho {rho.shape} {rho.dtype}   fvel {fvel.shape} {fvel.dtype}')
    print(f'{"":>14}  | cosmo [Om,Ob,h,ns,s8] = {np.round(cosmo, 4).tolist()}')
    assert rho.shape == (N, N, N), 'unexpected rho shape'
    assert fvel.shape == (N, N, N, 3), 'unexpected fvel shape'
    assert np.isfinite(rho).all() and np.isfinite(
        fvel).all(), 'non-finite values!'
print('\nAll runs present, shapes correct, no NaN/Inf.')

## 2. `rho` (overdensity δ) and `fvel` basic statistics

Expectations: δ has mean ≈ 0 (it is `rho/mean - 1`), min > -1 (no negative densities),
and a long positive tail (collapsed structures). Velocities are km/s, mean ≈ 0 per
component, with rms of order a few hundred km/s.

In [ ]:
print(f'{"run":>14} | {"d.mean":>9} {"d.min":>8} {"d.max":>9} {"d.std":>8} | {"v.mean":>8} {"v.rms":>8} {"|v|max":>9}')
for label, lhid in RUNS.items():
    aks = scale_factors(lhid)
    d = load_field(lhid, aks[-1], 'rho')
    v = load_field(lhid, aks[-1], 'fvel')
    vrms = np.sqrt((v ** 2).sum(-1)).std()
    vmag_max = np.sqrt((v ** 2).sum(-1)).max()
    print(f'{label:>14} | {d.mean():9.2e} {d.min():8.3f} {d.max():9.2f} {d.std():8.3f} | '
          f'{v.mean():8.2e} {v.std():8.1f} {vmag_max:9.1f}')
print('\nSanity: d.mean ~ 0, d.min > -1, d.std grows with structure; v.mean ~ 0, v.rms ~ 100s km/s.')

## 3. Density slices (final snapshot)

Visual check for cosmic web (filaments, voids, clusters) and absence of grid artefacts.
Shown as log10(1+δ) of a thin projected slab.

In [ ]:
fig, axes = plt.subplots(2, len(RUNS)//2, figsize=(5 * len(RUNS)/2, 5*2))
axes = axes.flatten()
for ax, (label, lhid) in zip(np.atleast_1d(axes), RUNS.items()):
    aks = scale_factors(lhid)
    d = load_field(lhid, aks[-1], 'rho')
    img = np.log10(np.clip(1 + projected_slab(d, thickness=8), 1e-2, None))
    im = ax.imshow(img.T, origin='lower', extent=[0, L, 0, L], cmap='inferno',
                   vmin=np.percentile(img, 1), vmax=np.percentile(img, 99.5))
    ax.set_title(f'{label}  (a={aks[-1]})')
    ax.set_xlabel('Mpc/h')
    plt.colorbar(im, ax=ax, fraction=0.046, label='log10(1+δ)')
plt.tight_layout()
plt.show()

## 4. One-point statistics: PDF of δ

Should be unimodal, peaked slightly below 0 (most volume is underdense voids), with a
long positive tail. log10(1+δ) makes the lognormal-ish shape easy to compare across runs.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
for label, lhid in RUNS.items():
    aks = scale_factors(lhid)
    d = load_field(lhid, aks[-1], 'rho').ravel()
    ax1.hist(d, bins=200, range=(-1, 5), histtype='step', density=True,
             color=COLORS[label], label=label)
    ax2.hist(np.log10(np.clip(1 + d, 1e-3, None)), bins=200, range=(-3, 1.5),
             histtype='step', density=True, color=COLORS[label], label=label)
ax1.set_xlabel('δ')
ax1.set_ylabel('PDF')
ax1.set_yscale('log')
ax1.legend()
ax2.set_xlabel('log10(1+δ)')
ax2.set_ylabel('PDF')
ax2.legend()
plt.tight_layout()
plt.show()

## 5. Two-point statistics: matter power spectrum P(k)

Should rise toward low k, turn over near the BAO/equality scale, and fall at high k.
Different cosmologies => different amplitudes/shapes, but all should look like a sane
nonlinear matter spectrum. The grey line marks the particle shot-noise floor.

In [ ]:
n_part = (N * 3) ** 3   # supersampling=3 -> 768^3 particles
shot = L ** 3 / n_part
k_nyq = np.pi * N / L

plt.figure(figsize=(7, 5.5))
for label, lhid in RUNS.items():
    aks = scale_factors(lhid)
    d = load_field(lhid, aks[-1], 'rho')
    k, pk = compute_pk(d)
    plt.loglog(k, pk, color=COLORS[label], label=label)
plt.axhline(shot, color='grey', ls=':', label=f'shot noise ~{shot:.1f}')
plt.axvline(k_nyq, color='grey', ls='--', alpha=0.5, label='Nyquist')
plt.xlabel('k [h/Mpc]')
plt.ylabel('P(k) [(Mpc/h)^3]')
plt.title('Matter power spectrum (final snapshot)')
plt.legend()
plt.tight_layout()
plt.show()

## 6. Velocity field sanity

Per-component velocity PDFs should be ~zero-mean and roughly Gaussian; the three
components should overlap (statistical isotropy).

In [ ]:
fig, axes = plt.subplots(1, len(RUNS), figsize=(5 * len(RUNS), 4), sharey=True)
for ax, (label, lhid) in zip(np.atleast_1d(axes), RUNS.items()):
    aks = scale_factors(lhid)
    v = load_field(lhid, aks[-1], 'fvel')
    for i, c in enumerate('xyz'):
        ax.hist(v[..., i].ravel(), bins=200, range=(-2000, 2000), histtype='step',
                density=True, label=f'v{c}')
    ax.set_title(label)
    ax.set_xlabel('v [km/s]')
    ax.legend(fontsize=8)
axes[0].set_ylabel('PDF') if len(RUNS) else None
plt.tight_layout()
plt.show()

## 7. Growth across snapshots (one run)

Structure should grow with time: P(k) amplitude and δ variance should increase
monotonically from the earliest to the latest scale factor.

In [ ]:
lhid = '2000'
aks = scale_factors(lhid)
plt.figure(figsize=(7, 5.5))
cmap = plt.cm.viridis(np.linspace(0, 1, len(aks)))
stds = []
for a_key, col in zip(aks, cmap):
    d = load_field(lhid, a_key, 'rho')
    stds.append(d.std())
    k, pk = compute_pk(d)
    plt.loglog(k, pk, color=col, label=f'a={a_key}')
plt.xlabel('k [h/Mpc]')
plt.ylabel('P(k) [(Mpc/h)^3]')
plt.title(f'Growth of P(k), lhid {lhid}')
plt.legend(fontsize=7)
plt.tight_layout()
plt.show()

print('std(δ) vs a (should increase monotonically):')
for a_key, s in zip(aks, stds):
    print(f'  a={a_key}: std={s:.4f}')
assert np.all(np.diff(stds) >
              0), 'WARNING: std(δ) not monotonically increasing!'
print('OK: structure grows monotonically with scale factor.')

## 8. Comparison against linear theory (CAMB)

The strongest normalization check. For each run we compute the **linear** matter P(k)
from CAMB at the **snapshot redshift** (z = 1/a - 1), using that run's own cosmology
(reuses `cmass.nbody.tools.get_camb_pk`, the same routine that seeds the ICs).

Expectations: FastPM (solid) should track linear theory (dashed) at large scales
(low k, ratio ≈ 1), then rise above it at high k from nonlinear growth. A low-k ratio
that is not ≈ 1 would flag a wrong amplitude (σ8 / growth / units) — catch it now.

In [ ]:
from cmass.nbody.tools import get_camb_pk

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for label, lhid in RUNS.items():
    aks = scale_factors(lhid)
    a = float(aks[-1])
    z = 1.0 / a - 1.0
    cosmo = load_cosmo(lhid)
    d = load_field(lhid, aks[-1], 'rho')
    k, pk = compute_pk(d)
    kcamb, pklin = get_camb_pk(np.asarray(k, float), *cosmo, z=z)
    ax1.loglog(k, pk, color=COLORS[label], label=f'{label} (z={z:.2f})')
    ax1.loglog(kcamb, pklin, color=COLORS[label], ls='--', alpha=0.7)
    ratio = pk / np.interp(k, kcamb, pklin)
    ax2.semilogx(k, ratio, color=COLORS[label], label=label)
ax1.axvline(k_nyq, color='grey', ls='--', alpha=0.4)
ax1.set_xlabel('k [h/Mpc]')
ax1.set_ylabel('P(k) [(Mpc/h)^3]')
ax1.set_title('solid = FastPM, dashed = CAMB linear')
ax1.legend(fontsize=8)
ax2.axhline(1.0, color='grey', ls=':')
ax2.axvline(k_nyq, color='grey', ls='--', alpha=0.4)
ax2.set_xlabel('k [h/Mpc]')
ax2.set_ylabel('P_FastPM / P_linear')
ax2.set_ylim(0, 3)
ax2.set_title('ratio (~1 at low k, >1 nonlinear)')
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

# Quantitative low-k check: median ratio over the few largest scales.
for label, lhid in RUNS.items():
    aks = scale_factors(lhid)
    a = float(aks[-1])
    z = 1.0 / a - 1.0
    cosmo = load_cosmo(lhid)
    k, pk = compute_pk(load_field(lhid, aks[-1], 'rho'))
    kcamb, pklin = get_camb_pk(np.asarray(k, float), *cosmo, z=z)
    lowk = (k > 0.01) & (k < 0.05)
    r = np.median(pk[lowk] / np.interp(k[lowk], kcamb, pklin))
    print(f'{label:>14}: median P_FastPM/P_lin over 0.01<k<0.05 = {r:.3f}')
print('\n(Expect ~1.0 +/- ~0.1; large-scale sample variance grows for few modes.)')